# K-fold CV results (`ogbench_kfold`)

Best-HP models re-run with stratified 5-fold CV. Aggregate across folds
(`dataset.split_params.data_seed` = 0..4) and plot overall test F1-macro.

**Outputs**
- `kfold_results.csv` — raw W&B runs
- `kfold_aggregated_results.csv` — 5-fold mean ± std per (dataset, model)
- `kfold_baseline_aggregated.csv` / `kfold_baseline_results.csv` — sklearn baselines
- `kfold_wilcoxon_signed_rank.csv` — paired Wilcoxon tests across folds
- `tutorials/plots/kfold_best_overall_test_f1_macro.{pdf,png}`

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.lines import Line2D
from scipy import stats

from table_generator import load_results_dataframe

ROOT = Path("..").resolve()
OUT_DIR = ROOT / "tutorials" / "plots"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load runs from W&B

In [ ]:
wandb_username = "bioshape-lab"
wandb_project = "ogbench_kfold"
csv_filename = str(ROOT / "kfold_results.csv")

df = load_results_dataframe(
    wandb_username,
    wandb_project,
    csv_filename=csv_filename,
    force_load=False,
    save_csv=True,
    batch_size=100,
    per_page=50,
    fetch_recent_only=False,
)

print(f"Loaded {len(df)} runs")
print(f"States: {df['state'].value_counts().to_dict() if 'state' in df.columns else 'n/a'}")

## 2. Aggregate across folds

Each `(model, dataset)` has one best-HP config × 5 folds. Fold-specific fields
(WGCNA `adjacency_threshold`, `num_nodes`, run paths, …) vary across folds, so
we aggregate explicitly by `(dataset, model)` rather than using
`aggregate_across_seeds`.

In [ ]:
df_finished = df[df["state"] == "finished"].copy() if "state" in df.columns else df.copy()
print(f"Finished runs: {len(df_finished)}")

df_finished["model"] = (
    df_finished["model.model_name"].astype(str).str.lower().str.strip("'")
)
df_finished["dataset"] = (
    df_finished["dataset.loader.parameters.data_name"].astype(str).str.strip("'")
)
df_finished["fold"] = pd.to_numeric(
    df_finished["dataset.split_params.data_seed"], errors="coerce"
)
df_finished["best_test_f1"] = pd.to_numeric(
    df_finished["summary.best_test/f1_macro"], errors="coerce"
)
df_finished["best_val_f1"] = pd.to_numeric(
    df_finished["summary.best_val/f1_macro"], errors="coerce"
)

counts = df_finished.groupby(["dataset", "model"]).size()
print("Runs per (dataset, model):")
print(counts.unstack(fill_value=0))
assert (counts == 5).all(), f"Expected 5 folds each, got:\n{counts[counts != 5]}"

aggregated_df = (
    df_finished.groupby(["dataset", "model"], as_index=False)
    .agg(
        best_test_f1_macro=("best_test_f1", "mean"),
        best_test_f1_macro_std=("best_test_f1", "std"),
        best_test_f1_macro_count=("best_test_f1", "count"),
        best_val_f1_macro=("best_val_f1", "mean"),
        best_val_f1_macro_std=("best_val_f1", "std"),
        best_val_f1_macro_count=("best_val_f1", "count"),
    )
)
agg_path = ROOT / "kfold_aggregated_results.csv"
aggregated_df.to_csv(agg_path, index=False)
print(f"▶ Saved aggregated results to: {agg_path}")
print(aggregated_df.shape)
aggregated_df.head()

## 3. Plot: Best Overall Model Performance by Test F1 Macro

SVM / Elastic Net lines are 5-fold mean test F1 from `ogbench_kfold`
(see `kfold_baseline_aggregated.csv`).

In [ ]:
plt.rcParams["font.family"] = "serif"
plt.rcParams["font.serif"] = ["CMU Serif", "DejaVu Serif", "Times New Roman"]

CANONICAL_MODEL_ORDER = [
    "mlp", "gps", "gin", "gcn", "gatv2", "sage", "sagn", "chebnet", "gatv4",
]
MODEL_DISPLAY_NAMES = {
    "mlp": "MLP", "gps": "GPS", "gin": "GIN", "gcn": "GCN", "gatv2": "GATv2",
    "sage": "SAGE", "sagn": "SAGN", "chebnet": "ChebNet", "gatv4": "MLA-GNN",
}
DATASET_ORDER = ["motrpac", "addneuromed", "parkinsons", "brca"]
DATASET_DISPLAY_NAMES = {
    "motrpac": "Heritage",
    "addneuromed": "Addneuromed",
    "parkinsons": "Parkinsons",
    "brca": "BRCA",
}
shared_model_colors_dict = {
    "mlp": "#5B9BD5",
    "gps": "#00B4D8",
    "gin": "#D62828",
    "gcn": "#E85D04",
    "gatv2": "#F48C06",
    "sage": "#FAA307",
    "sagn": "#7B2CBF",
    "chebnet": "#2D6A4F",
    "gatv4": "#52B788",
}
BASELINE_STYLES = {
    "svm": {"color": "black", "linestyle": "-", "linewidth": 2.5, "label": "SVM baseline"},
    "elastic_net": {
        "color": "black", "linestyle": ":", "linewidth": 3, "label": "Elastic Net baseline",
    },
}

# 5-fold mean test F1 from ogbench_kfold sklearn baselines
baseline_agg_path = ROOT / "kfold_baseline_aggregated.csv"
baseline_agg = pd.read_csv(baseline_agg_path)
BASELINES = {}
for _, row in baseline_agg.iterrows():
    BASELINES.setdefault(row["dataset"], {})[row["baseline"]] = float(row["test_f1_mean"])
print("K-fold baselines (test F1 mean):")
for ds in DATASET_ORDER:
    print(f"  {ds}: {BASELINES.get(ds)}")


def get_display_name(model):
    return MODEL_DISPLAY_NAMES.get(model, model)


def get_dataset_display_name(dataset):
    return DATASET_DISPLAY_NAMES.get(dataset, dataset.title())


df_plot = aggregated_df.copy()
assert (df_plot["best_test_f1_macro_count"] == 5).all()

models_in_data = set(df_plot["model"].dropna().unique().tolist())
all_models = [m for m in CANONICAL_MODEL_ORDER if m in models_in_data]
for m in sorted(models_in_data):
    if m not in all_models:
        all_models.append(m)

plot_data = {}
for model in all_models:
    plot_data[model] = {}
    for dataset in DATASET_ORDER:
        subset = df_plot[(df_plot["model"] == model) & (df_plot["dataset"] == dataset)]
        if len(subset) == 1:
            row = subset.iloc[0]
            plot_data[model][dataset] = {
                "mean": float(row["best_test_f1_macro"]),
                "std": float(row["best_test_f1_macro_std"]),
            }
        else:
            plot_data[model][dataset] = {"mean": np.nan, "std": np.nan}

print(f"Models: {[get_display_name(m) for m in all_models]}")
print(f"Datasets: {[get_dataset_display_name(d) for d in DATASET_ORDER]}")

In [ ]:
dataset_ylims = {}
for dataset in DATASET_ORDER:
    vals = []
    for model in all_models:
        data = plot_data[model][dataset]
        if not np.isnan(data["mean"]):
            std_val = data["std"] if not np.isnan(data["std"]) else 0.0
            vals.extend([data["mean"] - std_val, data["mean"] + std_val])
    for bvals in BASELINES.get(dataset, {}).values():
        vals.append(bvals)
    dataset_ylims[dataset] = (min(vals) - 0.05, max(vals) + 0.05) if vals else (0, 1)

fig, axes = plt.subplots(1, len(DATASET_ORDER), figsize=(16, 5))
fig.suptitle(
    "Best Overall Model Performance by Test F1 Macro",
    fontsize=26,
    fontweight="bold",
    y=0.995,
)
axes = np.atleast_1d(axes).reshape(1, -1)

for j, dataset in enumerate(DATASET_ORDER):
    ax = axes[0, j]
    means, stds, labels, colors = [], [], [], []
    for model in all_models:
        data = plot_data[model][dataset]
        if not np.isnan(data["mean"]):
            means.append(data["mean"])
            stds.append(data["std"] if not np.isnan(data["std"]) else 0.0)
            labels.append(get_display_name(model))
            colors.append(shared_model_colors_dict.get(model, "#888888"))

    if means:
        x_positions = np.arange(len(means))
        ax.bar(
            x_positions, means, yerr=stds, width=0.6, capsize=5,
            color=colors, edgecolor="black", linewidth=1.5,
        )
        if dataset in BASELINES:
            for baseline_name, baseline_value in BASELINES[dataset].items():
                style = BASELINE_STYLES[baseline_name]
                ax.axhline(
                    y=baseline_value,
                    color=style["color"],
                    linestyle=style["linestyle"],
                    linewidth=style["linewidth"],
                    alpha=0.9,
                )
        ax.set_xticks(x_positions)
        ax.set_xticklabels(labels, rotation=45, ha="right", fontsize=18)
    else:
        ax.set_xticks([])

    ax.tick_params(axis="both", labelsize=18)
    ax.set_title(get_dataset_display_name(dataset), fontsize=22, fontweight="bold", pad=10)
    if j == 0:
        ax.set_ylabel("F1 Macro Score", fontsize=20)
    ax.grid(axis="y", alpha=0.3, linestyle="--")
    ax.set_axisbelow(True)
    ax.set_ylim(dataset_ylims[dataset])

fig.legend(
    handles=[
        Line2D([0], [0], color="black", linestyle="-", linewidth=2.5, label="SVM baseline"),
        Line2D([0], [0], color="black", linestyle=":", linewidth=3, label="Elastic Net baseline"),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.93),
    ncol=2,
    fontsize=16,
    frameon=True,
)

plt.tight_layout(rect=[0, 0, 1, 0.92])
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

pdf_path = OUT_DIR / "kfold_best_overall_test_f1_macro.pdf"
png_path = OUT_DIR / "kfold_best_overall_test_f1_macro.png"
fig.savefig(pdf_path, bbox_inches="tight", dpi=300)
fig.savefig(png_path, bbox_inches="tight", dpi=300)
plt.show()
print(f"Saved {pdf_path}")
print(f"Saved {png_path}")

print("\nBest Test F1 Macro by Model and Dataset (5-fold mean ± std):")
print("=" * 80)
for model in all_models:
    print(f"\n{get_display_name(model)}:")
    for dataset in DATASET_ORDER:
        data = plot_data[model][dataset]
        if not np.isnan(data["mean"]):
            print(f"  {dataset}: {data['mean']:.4f} ± {data['std']:.4f}")
        else:
            print(f"  {dataset}: No data")

## 4. Wilcoxon signed-rank tests across folds

Paired tests on the **same 5 folds** (fold = `dataset.split_params.data_seed`).
For each dataset, compare fold-wise test F1 of each model against:

1. **SVM** (sklearn k-fold baseline)
2. **Elastic Net** (sklearn k-fold baseline)
3. **MLP** (same k-fold protocol as the GNNs)

With \(n=5\), power is limited; report the signed-rank statistic, two-sided \(p\),
mean paired \(\Delta\), and how many folds the left method wins.
Also pool all dataset×fold pairs (\(n=20\)) for an overall test.

In [ ]:
from scipy import stats

# Fold-level model scores (exclude any baseline rows that may be in the W&B CSV)
fold_models = df_finished[
    df_finished["model"].isin(CANONICAL_MODEL_ORDER)
    & df_finished["dataset"].isin(DATASET_ORDER)
][["dataset", "model", "fold", "best_test_f1"]].copy()
fold_models = fold_models.dropna(subset=["best_test_f1", "fold"])
fold_models["fold"] = fold_models["fold"].astype(int)

# Fold-level sklearn baselines
baseline_folds = pd.read_csv(ROOT / "kfold_baseline_results.csv")
baseline_folds = baseline_folds.rename(columns={"test_f1": "best_test_f1", "baseline": "model"})
baseline_folds = baseline_folds[["dataset", "model", "fold", "best_test_f1"]].copy()
baseline_folds["fold"] = baseline_folds["fold"].astype(int)

fold_scores = pd.concat([fold_models, baseline_folds], ignore_index=True)

# Wide: one column per method, index (dataset, fold)
wide = (
    fold_scores.pivot_table(
        index=["dataset", "fold"],
        columns="model",
        values="best_test_f1",
        aggfunc="first",
    )
    .sort_index()
)
print(f"Wide fold table: {wide.shape}  columns={list(wide.columns)}")
display(wide.head(10))


def wilcoxon_paired(left: np.ndarray, right: np.ndarray, alternative: str = "two-sided"):
    """Wilcoxon signed-rank on paired fold scores. Returns dict or None if not computable."""
    left = np.asarray(left, dtype=float)
    right = np.asarray(right, dtype=float)
    mask = np.isfinite(left) & np.isfinite(right)
    left, right = left[mask], right[mask]
    n = len(left)
    if n < 2:
        return None
    diff = left - right
    n_nonzero = int(np.sum(diff != 0))
    if n_nonzero < 1:
        return {
            "n": n,
            "n_nonzero": 0,
            "mean_delta": float(np.mean(diff)),
            "median_delta": float(np.median(diff)),
            "n_left_wins": int(np.sum(diff > 0)),
            "n_right_wins": int(np.sum(diff < 0)),
            "n_ties": int(np.sum(diff == 0)),
            "stat": np.nan,
            "p": np.nan,
            "alternative": alternative,
        }
    # SciPy: zero_method='wilcox' drops zero diffs; exact when possible
    kwargs = dict(alternative=alternative, zero_method="wilcox")
    try:
        res = stats.wilcoxon(left, right, method="exact", **kwargs)
    except TypeError:
        # Older SciPy without method=
        res = stats.wilcoxon(left, right, **kwargs)
    except ValueError:
        res = stats.wilcoxon(left, right, **kwargs)
    return {
        "n": n,
        "n_nonzero": n_nonzero,
        "mean_delta": float(np.mean(diff)),
        "median_delta": float(np.median(diff)),
        "n_left_wins": int(np.sum(diff > 0)),
        "n_right_wins": int(np.sum(diff < 0)),
        "n_ties": int(np.sum(diff == 0)),
        "stat": float(res.statistic),
        "p": float(res.pvalue),
        "alternative": alternative,
    }


COMPARATORS = ["svm", "elastic_net", "mlp"]
rows = []
for dataset in DATASET_ORDER:
    sub = wide.loc[dataset] if dataset in wide.index.get_level_values(0) else None
    if sub is None:
        continue
    for model in all_models:
        if model not in sub.columns:
            continue
        for comparator in COMPARATORS:
            if comparator == model or comparator not in sub.columns:
                continue
            pair = sub[[model, comparator]].dropna()
            out = wilcoxon_paired(pair[model].values, pair[comparator].values)
            if out is None:
                continue
            rows.append(
                {
                    "dataset": dataset,
                    "model": model,
                    "comparator": comparator,
                    "scope": "per_dataset",
                    **out,
                }
            )

# Pooled across datasets (same fold index paired within each dataset, then stacked)
for model in all_models:
    for comparator in COMPARATORS:
        if comparator == model:
            continue
        if model not in wide.columns or comparator not in wide.columns:
            continue
        pair = wide[[model, comparator]].dropna()
        out = wilcoxon_paired(pair[model].values, pair[comparator].values)
        if out is None:
            continue
        rows.append(
            {
                "dataset": "ALL",
                "model": model,
                "comparator": comparator,
                "scope": "pooled",
                **out,
            }
        )

wilcoxon_df = pd.DataFrame(rows)
wilcoxon_df["model_display"] = wilcoxon_df["model"].map(get_display_name)
wilcoxon_df["comparator_display"] = wilcoxon_df["comparator"].map(
    lambda x: {"svm": "SVM", "elastic_net": "Elastic Net", "mlp": "MLP"}.get(x, get_display_name(x))
)
wilcoxon_df["dataset_display"] = wilcoxon_df["dataset"].map(
    lambda x: "ALL" if x == "ALL" else get_dataset_display_name(x)
)

out_path = ROOT / "kfold_wilcoxon_signed_rank.csv"
wilcoxon_df.to_csv(out_path, index=False)
print(f"▶ Saved {out_path} ({len(wilcoxon_df)} tests)")


def _fmt_p(p):
    if pd.isna(p):
        return "—"
    if p < 1e-4:
        return f"{p:.2e}"
    return f"{p:.4f}"


def show_wilcoxon_table(comparator: str, scope: str = "per_dataset"):
    sub = wilcoxon_df[
        (wilcoxon_df["comparator"] == comparator) & (wilcoxon_df["scope"] == scope)
    ].copy()
    if sub.empty:
        print(f"No tests for {comparator} ({scope})")
        return
    label = {"svm": "SVM", "elastic_net": "Elastic Net", "mlp": "MLP"}[comparator]
    title_scope = "per dataset (n=5 folds)" if scope == "per_dataset" else "pooled (n≤20 dataset×folds)"
    print(f"\n{'=' * 88}")
    print(f"Wilcoxon signed-rank: model vs {label}  [{title_scope}]")
    print(f"Δ = model − {label}  on test F1-macro; two-sided p")
    print("=" * 88)

    if scope == "per_dataset":
        pivot_p = sub.pivot(index="model_display", columns="dataset_display", values="p")
        pivot_d = sub.pivot(index="model_display", columns="dataset_display", values="mean_delta")
        # Keep model order
        order = [get_display_name(m) for m in all_models if get_display_name(m) in pivot_p.index]
        cols = [get_dataset_display_name(d) for d in DATASET_ORDER if get_dataset_display_name(d) in pivot_p.columns]
        pivot_p = pivot_p.reindex(index=order, columns=cols)
        pivot_d = pivot_d.reindex(index=order, columns=cols)
        print("\nMean Δ (model − comparator):")
        display(pivot_d.map(lambda x: f"{x:+.4f}" if pd.notna(x) else "—"))
        print("Two-sided p-values:")
        display(pivot_p.map(_fmt_p))
    else:
        show = sub.sort_values("p")[
            [
                "model_display",
                "n",
                "mean_delta",
                "median_delta",
                "n_left_wins",
                "n_right_wins",
                "n_ties",
                "stat",
                "p",
            ]
        ].rename(
            columns={
                "model_display": "model",
                "mean_delta": "mean_Δ",
                "median_delta": "median_Δ",
                "n_left_wins": "model_wins",
                "n_right_wins": "comp_wins",
            }
        )
        show["mean_Δ"] = show["mean_Δ"].map(lambda x: f"{x:+.4f}")
        show["median_Δ"] = show["median_Δ"].map(lambda x: f"{x:+.4f}")
        show["p"] = show["p"].map(_fmt_p)
        display(show.reset_index(drop=True))


for comparator in COMPARATORS:
    show_wilcoxon_table(comparator, scope="per_dataset")

print("\n\n### Pooled across datasets")
for comparator in COMPARATORS:
    show_wilcoxon_table(comparator, scope="pooled")